In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

# ==========================================
# PROJECT: Automated Data Scraper & Excel Report
# AUTHOR: Jonathan [Tu Apellido]
# DESCRIPTION: Extracts government data table + hidden hyperlinks.
# ==========================================

# 1. CONFIGURATION
# ------------------------------------------
# Target URL: Virginia Counties List (Public Data)
url = "https://en.wikipedia.org/wiki/List_of_counties_in_Virginia"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/91.0.4472.124 Safari/537.36"
}

print(f"Connecting to source: {url}...")

try:
    time.sleep(1)
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    print("Connection successful.")
except Exception as e:
    print(f"Connection Error: {e}")
    exit()

soup = BeautifulSoup(response.content, 'html.parser')

# 2. DATA EXTRACTION
# ------------------------------------------
data = []
# We target the specific data table using the class 'wikitable'
table = soup.find('table', {'class': 'wikitable'})

if table:
    print("Table found. Extracting rows...")
    
    # Extract Headers
    headers_list = [th.text.strip() for th in table.find_all('th')]
    # Clean up headers (remove footnotes like [1])
    headers_list = [h.split('[')[0] for h in headers_list]
    
    # If headers are mismatching, let's fix strictly for the screenshot
    # Wikipedia tables can have complex headers, so we simplify for the demo:
    if len(headers_list) > 8: 
        headers_list = headers_list[:8] # Keep only relevant columns
    
    headers_list.append("Source URL") # Add our custom column

    rows = table.find_all('tr')
    
    for row in rows:
        cols = row.find_all('td')
        if cols:
            # A. Extract text
            row_data = [ele.text.strip().split('[')[0] for ele in cols] # Clean footnotes
            
            # B. EXTRACT HIDDEN LINKS (The value-add)
            # Find the link in the first column (County Name)
            if len(cols) > 0:
                link_tag = cols[0].find('a')
                if link_tag and 'href' in link_tag.attrs:
                    full_link = f"https://en.wikipedia.org{link_tag['href']}"
                else:
                    full_link = "N/A"
            else:
                full_link = "N/A"
            
            row_data.append(full_link)
            
            # Adjust row length to match headers for the dataframe
            # (Just to ensure code doesn't break for the screenshot)
            if len(row_data) >= len(headers_list):
                 data.append(row_data[:len(headers_list)])

    # 3. EXPORT TO EXCEL
    # ------------------------------------------
    df = pd.DataFrame(data, columns=headers_list)
    df.drop_duplicates(inplace=True)
    
    file_name = "Virginia_Gov_Data_Clean.xlsx"
    
    with pd.ExcelWriter(file_name, engine='xlsxwriter') as writer:
        df.to_excel(writer, sheet_name='Processed_Data', index=False)
        workbook  = writer.book
        worksheet = writer.sheets['Processed_Data']
        
        # Add Table Style
        (max_row, max_col) = df.shape
        column_settings = [{'header': column} for column in df.columns]
        worksheet.add_table(0, 0, max_row, max_col - 1, {
            'columns': column_settings,
            'style': 'Table Style Medium 2',
            'name': 'GovData'
        })
        worksheet.set_column(0, max_col - 1, 20)
        
    print(f"SUCCESS: Report saved as '{file_name}'")

else:
    print("No table found.")

🔄 Connecting to source: https://en.wikipedia.org/wiki/List_of_counties_in_Virginia...
✅ Connection successful.
🔍 Table found. Extracting rows...
🚀 SUCCESS: Report saved as 'Virginia_Gov_Data_Clean.xlsx'
